# 📡 Część 1: HTTP Server i WSGI - Od statyki do dynamiki

**Cel:** Zrozumieć czym jest HTTP server, jaki problem rozwiązuje WSGI i jak to działa.

**Analogia przewodnia:** 🍽️ Restauracja
- **HTTP Server** = kelner w bufecie (podaje gotowe rzeczy z półki)
- **WSGI** = tłumacz między kelnerem a kucharzem
- **Python app** = kucharz (gotuje na zamówienie)

---

## 1. HTTP Server - tradycyjny (statyczny)

### Czym jest HTTP Server?

**HTTP Server** (np. Apache, nginx w trybie statycznym) to program, który:

1. **Nasłuchuje** na porcie (zwykle 80 dla HTTP, 443 dla HTTPS)
2. **Odbiera** requesty od klientów (przeglądarek)
3. **Mapuje URL na pliki na dysku** ⬅️ KLUCZOWE!
4. **Zwraca** plik jako odpowiedź

### Przykład działania:

```
Klient → GET /about.html
           ↓
HTTP Server → sprawdza /var/www/html/about.html na dysku
           ↓
           zwraca zawartość pliku
           ↓
Klient ← otrzymuje HTML
```

### Struktura plików na serwerze:

```
/var/www/html/
├── index.html          → GET /
├── about.html          → GET /about.html
├── contact.html        → GET /contact.html
└── images/
    └── logo.png        → GET /images/logo.png
```

**URL mapuje się 1:1 na plik na dysku.**

---

### 🍽️ Analogia: Bufet (restauracja samoobsługowa)

Wyobraź sobie **bufet** w restauracji:

1. **Klient** przychodzi i mówi: "Poproszę kanapkę nr 3"
2. **Kelner** idzie do **konkretnej półki** (dysk)
3. **Bierze gotową kanapkę** nr 3 (plik HTML)
4. **Podaje klientowi**

**Wszystko jest z góry przygotowane i leży na półkach (statyczne pliki).**

```
Półka A → kanapka nr 1 (index.html)
Półka B → kanapka nr 2 (about.html)
Półka C → kanapka nr 3 (contact.html)
```

Kelner **nie gotuje**, tylko **podaje** to co już jest gotowe.

---

## 2. Problem: A co jeśli potrzebujemy DYNAMICZNOŚCI?

### Co jeśli chcemy:

❌ **Nie możemy zrobić HTTP Serverem (statycznym):**

1. **Pobrać dane z bazy danych**
   ```python
   # GET /api/users
   users = db.query(User).all()  # Zapytanie do PostgreSQL
   return users  # Zwróć jako JSON
   ```
   Tego **NIE MA** jako plik na dysku!

2. **Wykonać obliczenia**
   ```python
   # GET /api/stats
   total = sum([order.amount for order in orders])
   return {"total": total}
   ```
   Trzeba **przeliczyć** na żywo!

3. **Wygenerować HTML z danych użytkownika**
   ```python
   # GET /profile/{user_id}
   user = db.get_user(user_id)
   return f"<h1>Witaj {user.name}!</h1>"  # Dynamiczny HTML
   ```
   Każdy użytkownik ma **inną** stronę!

4. **Uruchomić kod Python**
   ```python
   # POST /api/orders
   send_email(user.email, "Order confirmed")
   return {"status": "ok"}
   ```

---

### 🍽️ Analogia: Klient chce danie "na zamówienie"

Klient przychodzi i mówi:
> "Poproszę spaghetti carbonara, ale bez cebuli, z dodatkiem boczku i ostrą papryczką"

**Problem:**
- Kelner **nie ma** takiego gotowego dania na półce
- To **NIE JEST** standardowa kanapka nr 3
- Trzeba to **przygotować NA ŻYWO**

**Rozwiązanie:**
- Kelner musi **poprosić kucharza** (Python kod)
- Kucharz **gotuje** na zamówienie (wykonuje kod)
- Zwraca gotowe danie (response)

**Ale jest problem: kelner mówi po polsku, a kucharz po "kuchennym" (Python)!**

**Potrzebujemy TŁUMACZA!** ⬅️ To jest WSGI

---

## 3. WSGI - tłumacz między HTTP a Python

### Czym jest WSGI?

**WSGI** = **Web Server Gateway Interface**

To **standard/protokół** (NIE program!), który opisuje:
- **Jak HTTP server ma rozmawiać z aplikacją Python**
- **W jakim formacie przekazywać dane**
- **Jak odbierać odpowiedzi**

### Problem, który rozwiązuje WSGI:

```
HTTP Server mówi po "HTTP-owym":
  GET /api/users HTTP/1.1
  Host: example.com
  User-Agent: Mozilla/5.0
  ...

      ❌ NIE ROZUMIEJĄ SIĘ ❌

Python application mówi po "Pythonowym":
  def get_users():
      return {"users": [...]}
```

**WSGI to wspólny język/protokół komunikacji!**

---

### Jak działa WSGI - krok po kroku:

```
1. HTTP Request (GET /api/users)
        ↓
2. HTTP Server odbiera request
        ↓
3. WSGI Server (np. Gunicorn) - tłumaczy na Python:
        ↓
   Wywołuje: app(environ, start_response)
   
   environ = {  # Dict z informacjami o requeście
       'REQUEST_METHOD': 'GET',
       'PATH_INFO': '/api/users',
       'HTTP_HOST': 'example.com',
       ...
   }
        ↓
4. Twoja aplikacja Python (Django/Flask) wykonuje kod:
        ↓
   users = db.query(User).all()
   return [{"id": 1, "name": "John"}, ...]
        ↓
5. WSGI Server tłumaczy z powrotem na HTTP:
        ↓
   HTTP/1.1 200 OK
   Content-Type: application/json
   
   [{"id": 1, "name": "John"}]
        ↓
6. HTTP Response wraca do klienta
```

---

### 🍽️ Analogia: Kelner + Tłumacz + Kucharz

```
1. KLIENT (po polsku):
   "Poproszę carbonara bez cebuli"
        ↓
2. KELNER (mówi po polsku, ale nie gotuje):
   Słyszy zamówienie
        ↓
3. TŁUMACZ WSGI przekłada na "język kuchenny":
   dish = "carbonara"
   ingredients = {"bacon": True, "onion": False}
        ↓
4. KUCHARZ (Python app):
   Rozumie i gotuje danie
   Zwraca: 🍝 gotowe carbonara
        ↓
5. TŁUMACZ WSGI przekłada z powrotem:
   "Carbonara gotowa!"
        ↓
6. KELNER podaje klientowi
```

**Tłumacz WSGI umożliwia komunikację między kelnerem (HTTP) a kucharzem (Python).**

---

## 4. WSGI w praktyce - kod

### Najprostsza WSGI aplikacja:


In [ ]:
# Najprostsza możliwa WSGI aplikacja

def application(environ, start_response):
    """
    WSGI application - to jest interfejs WSGI!
    
    Parametry:
    - environ: dict z informacjami o requeście (URL, headers, metoda HTTP, etc.)
    - start_response: funkcja do ustawienia statusu i headers w odpowiedzi
    
    Zwraca:
    - iterable of bytes (zawartość odpowiedzi)
    """
    
    # 1. Ustaw status i headers
    status = '200 OK'
    headers = [('Content-Type', 'text/plain')]
    start_response(status, headers)
    
    # 2. Zwróć response body jako iterable of bytes
    return [b"Hello from WSGI!"]

# Tę funkcję application() wywołuje WSGI server (Gunicorn/uWSGI)
# dla KAŻDEGO HTTP requesta

### Bardziej realistyczny przykład:

In [ ]:
import json

def application(environ, start_response):
    """
    WSGI app z routingiem i JSON response
    """
    
    # Odczytaj ścieżkę z requesta
    path = environ.get('PATH_INFO', '/')
    method = environ.get('REQUEST_METHOD', 'GET')
    
    # Prosty routing
    if path == '/' and method == 'GET':
        status = '200 OK'
        headers = [('Content-Type', 'application/json')]
        start_response(status, headers)
        
        response_data = {"message": "Welcome to WSGI API"}
        return [json.dumps(response_data).encode('utf-8')]
    
    elif path == '/api/users' and method == 'GET':
        status = '200 OK'
        headers = [('Content-Type', 'application/json')]
        start_response(status, headers)
        
        # Symulacja pobrania z bazy (w rzeczywistości byłoby db.query)
        users = [
            {"id": 1, "name": "Alice"},
            {"id": 2, "name": "Bob"}
        ]
        return [json.dumps(users).encode('utf-8')]
    
    else:
        # 404 Not Found
        status = '404 Not Found'
        headers = [('Content-Type', 'text/plain')]
        start_response(status, headers)
        return [b"404 - Not Found"]

# To jest cała aplikacja!
# WSGI server (Gunicorn) wywołuje tę funkcję dla każdego requesta

### Jak uruchomić WSGI aplikację?

```bash
# Zapisz powyższy kod jako: app.py

# Uruchom z Gunicorn (WSGI server):
gunicorn app:application

# Aplikacja dostępna na: http://localhost:8000
```

**Gunicorn to implementacja WSGI servera** - program, który:
1. Odbiera HTTP requesty
2. Przekształca je na wywołanie `application(environ, start_response)`
3. Odbiera response z aplikacji
4. Przekształca z powrotem na HTTP response

---

## 5. WSGI w Django i Flask

### Django

Django **jest aplikacją WSGI** (ma wbudowaną funkcję `application`):

```python
# Django automatycznie tworzy plik: myproject/wsgi.py

import os
from django.core.wsgi import get_wsgi_application

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'myproject.settings')

application = get_wsgi_application()  # To jest WSGI application!
```

**Development:**
```bash
python manage.py runserver
# Używa wbudowanego prostego WSGI servera (tylko dev!)
```

**Production:**
```bash
gunicorn myproject.wsgi:application --workers 4
# Gunicorn jako production WSGI server
```

---

### Flask

Flask też jest aplikacją WSGI:

```python
# app.py
from flask import Flask

app = Flask(__name__)  # app to WSGI application!

@app.route('/')
def hello():
    return "Hello World"
```

**Development:**
```bash
flask run
# Używa wbudowanego Werkzeug WSGI servera (tylko dev!)
```

**Production:**
```bash
gunicorn app:app --workers 4
# Gunicorn jako production WSGI server
```

---

## 6. Popularne WSGI Servery

**WSGI to standard, ale potrzebujesz programu, który go implementuje:**

### Gunicorn ⭐ (najpopularniejszy)
- **Green Unicorn**
- Prosty, szybki, production-ready
- Pre-fork worker model (wiele procesów)
- Domyślny dla większości Django/Flask deployments

```bash
gunicorn myapp:application --workers 4 --bind 0.0.0.0:8000
```

### uWSGI
- Bardzo wydajny, ale bardziej skomplikowany
- Wiele opcji konfiguracji
- Używany z nginx

```bash
uwsgi --http :8000 --wsgi-file myapp.py --callable application
```

### mod_wsgi (dla Apache)
- Moduł Apache HTTP Server
- Integruje Python bezpośrednio z Apache

### Waitress
- Pure-Python WSGI server
- Działa na Windows (Gunicorn nie)
- Mniej wydajny, ale prostszy

```python
from waitress import serve
serve(application, host='0.0.0.0', port=8000)
```

---

## 7. Podsumowanie: HTTP Server → WSGI → Python App

### Przed WSGI (tylko statyczne pliki):

```
Klient → HTTP Server (Apache/nginx) → Plik z dysku → Klient
```

❌ Nie można wykonywać kodu Python
❌ Nie można łączyć się z bazą danych
❌ Nie można generować dynamicznego contentu

---

### Po WSGI (dynamiczne aplikacje):

```
Klient → HTTP Server → WSGI Server (Gunicorn) → Python App (Django/Flask) → Baza danych
                         ↑                         ↑
                      Tłumacz                   Twój kod
```

✅ Możesz wykonywać kod Python
✅ Możesz łączyć się z bazą danych
✅ Możesz generować dynamiczny content
✅ Możesz robić API, autentykację, logikę biznesową

---

### 🍽️ Analogia finalna:

**Przed WSGI:**
- Kelner → Półka z gotowymi kanapkami → Klient
- Tylko bufet!

**Po WSGI:**
- Kelner → Tłumacz → Kucharz → Gotuje na zamówienie → Klient
- Pełna restauracja!

---

## 🎯 Kluczowe wnioski:

1. **HTTP Server** = serwuje pliki z dysku (statyczne)
2. **WSGI** = standard komunikacji HTTP ↔ Python (dla dynamicznych aplikacji)
3. **WSGI Server** (Gunicorn, uWSGI) = program implementujący standard WSGI
4. **Django/Flask** = aplikacje WSGI (Twój kod)

**WSGI umożliwia uruchamianie kodu Python w odpowiedzi na HTTP requesty.**

---

## 📚 Dalej:

**Następny notebook:** `02_wsgi_vs_asgi.ipynb`
- Czym jest ASGI?
- Co DOKŁADNIE jest asynchroniczne?
- Czym różni się od WSGI?
- Kiedy używać ASGI zamiast WSGI?

---